<a href="https://colab.research.google.com/github/tinana2k/CS-5567---Deep-Learning/blob/main/HW3_Transformer/HW3_formal_Tina_Nguyen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework 3 - Attention and Transformer
- Classify the speakers of given features.
- Main goal: Learn how to use transformer.

# Data

## Dataset
- Original dataset is [Voxceleb1](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/).
- The [license](https://creativecommons.org/licenses/by/4.0/) and [complete version](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/files/license.txt) of Voxceleb1.
- We randomly select 600 speakers from Voxceleb1.
- Then preprocess the raw waveforms into mel-spectrograms.

- Args:
  - data_dir: The path to the data directory.
  - metadata_path: The path to the metadata.
  - segment_len: The length of audio segment for training.
- The architecture of data directory \\
  - data directory \\
  |---- metadata.json \\
  |---- testdata.json \\
  |---- mapping.json \\
  |---- uttr-{random string}.pt \\

- The information in metadata
  - "n_mels": The dimention of mel-spectrogram.
  - "speakers": A dictionary.
    - Key: speaker ids.
    - value: "feature_path" and "mel_len"


For efficiency, we segment the mel-spectrograms into segments in the traing step.

In [ ]:

!unzip path_to/Dataset.zip

# **Self-code**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls -lh "/content/drive/MyDrive/CS5567/HW3/"

total 7.6G
-rw------- 1 root root 7.6G Sep 29 21:15 Dataset_HW3.zip


In [3]:
!mkdir -p /content/Dataset_HW3
!unzip -q "/content/drive/MyDrive/CS5567/HW3/Dataset_HW3.zip" -d "/content/Dataset_HW3"

In [4]:
!ls -lh /content/Dataset_HW3

total 5.0M
drwxr-xr-x 2 root root 4.9M Mar 19  2021 Dataset


In [5]:
!du -sh /content/Dataset_HW3

8.5G	/content/Dataset_HW3


In [6]:
!find /content/Dataset_HW3 -maxdepth 2 | head -30

/content/Dataset_HW3
/content/Dataset_HW3/Dataset
/content/Dataset_HW3/Dataset/uttr-f2e2dc9d3a6c471abd6464a316d1e21a.pt
/content/Dataset_HW3/Dataset/uttr-d6283f2c24b84aa5881b27415769f217.pt
/content/Dataset_HW3/Dataset/uttr-876cfe8be8af42c8b44db61f18dc116e.pt
/content/Dataset_HW3/Dataset/uttr-10b85e1ea19840db890b9ad577f2d340.pt
/content/Dataset_HW3/Dataset/uttr-1d8378c98a1b4b239092c4e5e29155f8.pt
/content/Dataset_HW3/Dataset/uttr-b3af2e2e4a144833bec132a241397df6.pt
/content/Dataset_HW3/Dataset/uttr-b3cd7b36c9304129ac428a3b4cea71bf.pt
/content/Dataset_HW3/Dataset/uttr-2c835c0439954345a45dd6cb35454132.pt
/content/Dataset_HW3/Dataset/uttr-e2edd8f4e0d748918c81d26d76fae0cc.pt
/content/Dataset_HW3/Dataset/uttr-8b5c42c6d9f9484cb0062ea6230e09ba.pt
/content/Dataset_HW3/Dataset/uttr-d58cca4648274afdb7ca7249e80eff7d.pt
/content/Dataset_HW3/Dataset/uttr-84895affd5c048d7832d382724a1e6ad.pt
/content/Dataset_HW3/Dataset/uttr-22c7da18451a4de083f47bded5606b46.pt
/content/Dataset_HW3/Dataset/uttr-cf49d4

In [7]:
!find /content/Dataset_HW3 -name "metadata.json"
!find /content/Dataset_HW3 -name "mapping.json"

/content/Dataset_HW3/Dataset/metadata.json
/content/Dataset_HW3/Dataset/mapping.json


In [8]:
data_dir = "/content/Dataset_HW3"

In [9]:
data_dir = "/content/Dataset_HW3/Dataset"

# **Back to the lesson:**

In [10]:
import os
import json
import torch
import random
from pathlib import Path
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence


class myDataset(Dataset):
  def __init__(self, data_dir, segment_len=128):
    self.data_dir = data_dir
    self.segment_len = segment_len

    # Load the mapping from speaker neme to their corresponding id.
    mapping_path = Path(data_dir) / "mapping.json"
    mapping = json.load(mapping_path.open())
    self.speaker2id = mapping["speaker2id"]

    # Load metadata of training data.
    metadata_path = Path(data_dir) / "metadata.json"
    metadata = json.load(open(metadata_path))["speakers"]

    # Get the total number of speaker.
    self.speaker_num = len(metadata.keys())
    self.data = []
    for speaker in metadata.keys():
      for utterances in metadata[speaker]:
        self.data.append([utterances["feature_path"], self.speaker2id[speaker]])

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    feat_path, speaker = self.data[index]
    # Load preprocessed mel-spectrogram.
    mel = torch.load(os.path.join(self.data_dir, feat_path))

    # Segmemt mel-spectrogram into "segment_len" frames.
    if len(mel) > self.segment_len:
      # Randomly get the starting point of the segment.
      start = random.randint(0, len(mel) - self.segment_len)
      # Get a segment with "segment_len" frames.
      mel = torch.FloatTensor(mel[start:start+self.segment_len])
    else:
      mel = torch.FloatTensor(mel)
    # Turn the speaker id into long for computing loss later.
    speaker = torch.FloatTensor([speaker]).long()
    return mel, speaker

  def get_speaker_number(self):
    return self.speaker_num

## Dataloader (2pts)
- Split dataset into training dataset(90%) and validation dataset(10%).
- Create dataloader to iterate the data.


In [11]:
import torch
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence


def collate_batch(batch):
  # Process features within a batch.
  """Collate a batch of data."""
  mel, speaker = zip(*batch)
  # Because we train the model batch by batch, we need to pad the features in the same batch to make their lengths the same.
  mel = pad_sequence(mel, batch_first=True, padding_value=-20)    # pad log 10^(-20) which is very small value.
  # mel: (batch size, length, 40)
  return mel, torch.FloatTensor(speaker).long()


def get_dataloader(data_dir, batch_size, n_workers):
  """Generate dataloader"""
  dataset = myDataset(data_dir)
  speaker_num = dataset.get_speaker_number()
  # Split dataset into training dataset and validation dataset
  trainlen = int(0.9 * len(dataset))
  lengths = [trainlen, len(dataset) - trainlen]
  trainset, validset = random_split(dataset, lengths)

# (1pt)TODO: Define Training Data Loader and set batch_size=batch_size,shuffle=True,drop_last=True,num_workers=n_workers,pin_memory=True and collate_fn=collate_batch
  train_loader = DataLoader(
      trainset,
      batch_size=batch_size,shuffle=True,drop_last=True,num_workers=n_workers,pin_memory=False, collate_fn=collate_batch # My Macbook M4 GPU doesn't support pin_memory so I had turned it off
  )
  # (1pt)TODO: Define Validation Data Loader and set batch_size=batch_size,drop_last=True,num_workers=n_workers,pin_memory=True and collate_fn=collate_batch
  valid_loader = DataLoader(
      validset,
      batch_size=batch_size,drop_last=True,num_workers=n_workers,pin_memory=False, collate_fn=collate_batch # My Macbook M4 GPU doesn't support pin_memory so I had turned it off
  )

  return train_loader, valid_loader, speaker_num

# Model (5pts)
- TransformerEncoderLayer:
  - Base transformer encoder layer in [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
  - Parameters:
    - d_model: the number of expected features of the input (required).

    - nhead: the number of heads of the multiheadattention models (required).

    - dim_feedforward: the dimension of the feedforward network model (default=2048).

    - dropout: the dropout value (default=0.1).

    - activation: the activation function of intermediate layer, relu or gelu (default=relu).

- TransformerEncoder:
  - TransformerEncoder is a stack of N transformer encoder layers
  - Parameters:
    - encoder_layer: an instance of the TransformerEncoderLayer() class (required).

    - num_layers: the number of sub-encoder-layers in the encoder (required).

    - norm: the layer normalization component (optional).

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Classifier(nn.Module):
  def __init__(self, d_model=80, n_spks=600, dropout=0.1):
    super().__init__()
    # Project the dimension of features from that of input into d_model.
    self.prenet = nn.Linear(40, d_model) # (1pt)TODO: Linear layer from 40 to d_model

    # (1pt)TODO: Use TransformerEncoderLayer as encoder_layer and set dim_feedforward=256 and nhead=2
    self.encoder_layer = nn.TransformerEncoderLayer(
        d_model=d_model, dim_feedforward=256, nhead=2
    )
    self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=2)

    # Project the the dimension of features from d_model into speaker nums.
    self.pred_layer = nn.Sequential(
      nn.Linear(d_model, d_model), # (1pt)TODO: Linear layer from d_model to d_model
      nn.ReLU(), # (1pt)TODO: Relu layer
      nn.Linear(d_model, n_spks), # (1pt)TODO: Linear layer from d_model to n_spks
    )

  def forward(self, mels):
    """
    args:
      mels: (batch size, length, 40)
    return:
      out: (batch size, n_spks)
    """
    # out: (batch size, length, d_model)
    out = self.prenet(mels)
    # out: (length, batch size, d_model)
    out = out.permute(1, 0, 2)
    # The encoder layer expect features in the shape of (length, batch size, d_model).
    out = self.encoder(out)
    # out: (batch size, length, d_model)
    out = out.transpose(0, 1)
    # mean pooling
    stats = out.mean(dim=1)

    # out: (batch, n_spks)
    out = self.pred_layer(stats)
    return out

# Learning rate schedule
- For transformer architecture, the design of learning rate schedule is different from that of CNN.
- Previous works show that the warmup of learning rate is useful for training models with transformer architectures.
- The warmup schedule
  - Set learning rate to 0 in the beginning.
  - The learning rate increases linearly from 0 to initial learning rate during warmup period.

In [13]:
import math

import torch
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LambdaLR


def get_cosine_schedule_with_warmup(
  optimizer: Optimizer,
  num_warmup_steps: int,
  num_training_steps: int,
  num_cycles: float = 0.5,
  last_epoch: int = -1,
):
  """
  Create a schedule with a learning rate that decreases following the values of the cosine function between the
  initial lr set in the optimizer to 0, after a warmup period during which it increases linearly between 0 and the
  initial lr set in the optimizer.

  Args:
    optimizer (:class:`~torch.optim.Optimizer`):
      The optimizer for which to schedule the learning rate.
    num_warmup_steps (:obj:`int`):
      The number of steps for the warmup phase.
    num_training_steps (:obj:`int`):
      The total number of training steps.
    num_cycles (:obj:`float`, `optional`, defaults to 0.5):
      The number of waves in the cosine schedule (the defaults is to just decrease from the max value to 0
      following a half-cosine).
    last_epoch (:obj:`int`, `optional`, defaults to -1):
      The index of the last epoch when resuming training.

  Return:
    :obj:`torch.optim.lr_scheduler.LambdaLR` with the appropriate schedule.
  """

  def lr_lambda(current_step):
    # Warmup
    if current_step < num_warmup_steps:
      return float(current_step) / float(max(1, num_warmup_steps))
    # decadence
    progress = float(current_step - num_warmup_steps) / float(
      max(1, num_training_steps - num_warmup_steps)
    )
    return max(
      0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))
    )

  return LambdaLR(optimizer, lr_lambda, last_epoch)


# Model Function (1pt)
- Model forward function.

In [14]:
import torch


def model_fn(batch, model, criterion, device):
  """Forward a batch through the model."""

  mels, labels = batch
  mels = mels.to(device)
  labels = labels.to(device)

  outs = model(mels)

  loss = criterion(outs, labels) #(1pt)TODO: Use criterion to calculate the loss

  # Get the speaker id with highest probability.
  preds = outs.argmax(1)
  # Compute accuracy.
  accuracy = torch.mean((preds == labels).float())

  return loss, accuracy

# Validate (3pts)
- Calculate accuracy of the validation set.

In [15]:
from tqdm import tqdm
import torch


def valid(dataloader, model, criterion, device):
  """Validate on validation set."""

  model.eval()
  running_loss = 0.0
  running_accuracy = 0.0
  pbar = tqdm(total=len(dataloader.dataset), ncols=0, desc="Valid", unit=" uttr")
  for i, batch in enumerate(dataloader):
    with torch.no_grad():
      loss, accuracy = model_fn(batch, model, criterion,device) #(1pt)TODO: Use model_fn function above to calculate the loss and accuracy
      running_loss += loss.item() #(1pt)TODO: Get cumulative loss
      running_accuracy += accuracy.item()   #(1pt)TODO: Get cumulative Accuracy

    pbar.update(dataloader.batch_size)
    pbar.set_postfix(
      loss=f"{running_loss / (i+1):.2f}",
      accuracy=f"{running_accuracy / (i+1):.2f}",
    )

  pbar.close()
  model.train()

  return running_accuracy / len(dataloader)

# Main function (5pts)

In [18]:
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split


def parse_args():
  """arguments"""
  config = {
    "data_dir": "/content/Dataset_HW3/Dataset",
    "save_path": "model.ckpt",
    "batch_size": 32,
    "n_workers": 0,
    "valid_steps": 2000,
    "warmup_steps": 1000,
    "save_steps": 10000,
    "total_steps": 20000,
  }

  return config


def main(
  data_dir,
  save_path,
  batch_size,
  n_workers,
  valid_steps,
  warmup_steps,
  total_steps,
  save_steps,
):
  """Main function."""

  if torch.cuda.is_available():
      device = torch.device("cuda")
  elif torch.backends.mps.is_available():
      device = torch.device("mps")
  else:
      device = torch.device("cpu")

  print(f"[Info]: Use {device} now!")

  train_loader, valid_loader, speaker_num = get_dataloader(
      data_dir,
      batch_size,
      n_workers
  )

  train_iterator = iter(train_loader)

  print("[Info]: Finish loading data!", flush=True)

  model = Classifier(n_spks=speaker_num).to(device)

  criterion = nn.CrossEntropyLoss()

  optimizer = AdamW(
      model.parameters(),
      lr=1e-3
  )

  scheduler = get_cosine_schedule_with_warmup(
      optimizer,
      warmup_steps,
      total_steps
  )

  print("[Info]: Finish creating model!", flush=True)

  best_accuracy = -1.0
  best_state_dict = None

  pbar = tqdm(
      total=valid_steps,
      ncols=0,
      desc="Train",
      unit=" step"
  )

  for step in range(total_steps):

    try:
      batch = next(train_iterator)

    except StopIteration:
      train_iterator = iter(train_loader)
      batch = next(train_iterator)

    loss, accuracy = model_fn(
        batch,
        model,
        criterion,
        device
    )

    batch_loss = loss.item()
    batch_accuracy = accuracy.item()

    # Update model
    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    # Log
    pbar.update()

    pbar.set_postfix(
      loss=f"{batch_loss:.2f}",
      accuracy=f"{batch_accuracy:.2f}",
      step=step + 1,
    )

    # Validation
    if (step + 1) % valid_steps == 0:

      pbar.close()

      valid_accuracy = valid(
          valid_loader,
          model,
          criterion,
          device
      )

      if valid_accuracy > best_accuracy:
        best_accuracy = valid_accuracy

        # safer copy of best model
        best_state_dict = {
            k: v.cpu().clone()
            for k, v in model.state_dict().items()
        }

      pbar = tqdm(
          total=valid_steps,
          ncols=0,
          desc="Train",
          unit=" step"
      )

    # Save model
    if (step + 1) % save_steps == 0 and best_state_dict is not None:

      torch.save(
          best_state_dict,
          save_path
      )

      pbar.write(
          f"Step {step + 1}, best model saved. "
          f"(accuracy={best_accuracy:.4f})"
      )

  pbar.close()


if __name__ == "__main__":
  main(**parse_args())

[Info]: Use cuda now!
[Info]: Finish loading data!


/tmp/ipykernel_1276/870100952.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=2)


[Info]: Finish creating model!


Train: 100% 2000/2000 [00:53<00:00, 37.51 step/s, accuracy=0.19, loss=4.06, step=2000]
Valid: 100% 6944/6944 [00:04<00:00, 1502.86 uttr/s, accuracy=0.22, loss=3.75]
Train: 100% 2000/2000 [00:51<00:00, 38.49 step/s, accuracy=0.19, loss=3.21, step=4000]
Valid: 100% 6944/6944 [00:04<00:00, 1584.85 uttr/s, accuracy=0.33, loss=3.03]
Train: 100% 2000/2000 [00:51<00:00, 39.07 step/s, accuracy=0.56, loss=1.65, step=6000]
Valid: 100% 6944/6944 [00:04<00:00, 1591.03 uttr/s, accuracy=0.41, loss=2.66]
Train: 100% 2000/2000 [00:50<00:00, 39.71 step/s, accuracy=0.50, loss=2.02, step=8000]
Valid: 100% 6944/6944 [00:04<00:00, 1490.24 uttr/s, accuracy=0.45, loss=2.43]
Train: 100% 2000/2000 [00:52<00:00, 38.25 step/s, accuracy=0.56, loss=1.88, step=1e+4]
Valid: 100% 6944/6944 [00:04<00:00, 1524.48 uttr/s, accuracy=0.49, loss=2.25]
Train:   0% 7/2000 [00:00<00:53, 36.92 step/s, accuracy=0.53, loss=2.24, step=1e+4]

Step 10000, best model saved. (accuracy=0.4859)


Train: 100% 2000/2000 [00:52<00:00, 38.23 step/s, accuracy=0.50, loss=1.67, step=12000]
Valid: 100% 6944/6944 [00:04<00:00, 1555.08 uttr/s, accuracy=0.53, loss=2.06]
Train: 100% 2000/2000 [00:51<00:00, 38.51 step/s, accuracy=0.66, loss=1.49, step=14000]
Valid: 100% 6944/6944 [00:04<00:00, 1552.06 uttr/s, accuracy=0.57, loss=1.89]
Train: 100% 2000/2000 [00:52<00:00, 38.36 step/s, accuracy=0.47, loss=2.14, step=16000]
Valid: 100% 6944/6944 [00:04<00:00, 1538.85 uttr/s, accuracy=0.57, loss=1.83]
Train: 100% 2000/2000 [00:51<00:00, 38.88 step/s, accuracy=0.62, loss=1.51, step=18000]
Valid: 100% 6944/6944 [00:04<00:00, 1557.82 uttr/s, accuracy=0.59, loss=1.79]
Train: 100% 2000/2000 [00:52<00:00, 38.05 step/s, accuracy=0.69, loss=1.37, step=2e+4]
Valid: 100% 6944/6944 [00:04<00:00, 1501.00 uttr/s, accuracy=0.60, loss=1.75]
Train:   0% 0/2000 [00:00<?, ? step/s]

Step 20000, best model saved. (accuracy=0.5988)
